In [24]:
from utils.regex_utils import snake_case_to_pascal_case, snake_case_to_words
from utils.loader import load_json_folder, load_json
from neo4j_manager import Neo4jManager
from config import get_settings
from typing import Optional
import os


In [25]:
metada_dir = "./metadata"
structure_dir = os.path.join(metada_dir, "structure")
collections_dir = os.path.join(metada_dir, "collections")


In [26]:
settings = get_settings()
print(settings)


neo4j_uri='neo4j://localhost:7687' neo4j_user='neo4j' neo4j_password='hola123hola123' ollama_base_url='http://localhost:11434' ollama_model='llama3.1:8b' ollama_embedding_model='nomic-embed-text'


In [27]:
manager = Neo4jManager()

manager.execute_query("MATCH (n) DETACH DELETE n")

# manager.create_constraints()


[]

In [28]:
collections = load_json_folder(collections_dir)

for label, collection in collections.items():
	label = snake_case_to_pascal_case(label)

	query = f"""
	MERGE (n:{label} {{name: $name}})
	SET n.description = $description
	"""

	for name, description in collection.items():
		# name = snake_case_to_words(name)

		manager.execute_query(query, {
			"name": name,
			"description": description
		})


In [29]:
def insert_node(label: str, name: str, node: dict, parent: Optional[str]=None):
	query = f"""
	MERGE (n:{label} {{name: $name}})
	SET n.description = $description
	WITH n
	OPTIONAL MATCH (p:{label} {{name: $parent}})
	FOREACH (_ IN CASE WHEN p IS NULL THEN [] ELSE [1] END |
		MERGE (p)-[:HAS_CHILD]->(n)
	)
	"""	

	# name = snake_case_to_words(name)

	params = {
		"name": name,
		"description": node.get("description"),
		"parent": parent
	}

	manager.execute_query(query, params)
	
	for child_name, child_node in node.get("children", {}).items():
		insert_node(label, child_name, child_node, name)


In [30]:
structures = load_json_folder(structure_dir)


In [31]:
for node in structures.values():
	label = next(iter(node))
	root_node = node[label]

	insert_node(snake_case_to_pascal_case(label), label, root_node)


In [32]:
folktale = load_json("./2_sharing_joy_and_sorrow.json")
# print(folktale)


In [33]:
query = """
MERGE (f:Folktale {url: $url})
SET f.title = $title,
    f.summary = $summary

WITH f
MATCH (g:Genre {name: $genre})
MERGE (f)-[:HAS_GENRE]->(g)

WITH f
MATCH (n:Nation {name: $nation})
MERGE (f)-[:FROM_NATION]->(n)

RETURN f
"""

params = {
    "url": folktale["url"],
    "title": folktale["title"],
    "summary": folktale["summary"],
    "genre": folktale["genre"],
    "nation": folktale["nation"]
}

manager.execute_query(query, params)


[{'f': {'summary': "A quarrelsome tailor constantly abused his kind and hardworking wife. Authorities imprisoned him to reform his behavior, and he promised to treat her well. However, he soon resumed his abusive ways, this time tearing her hair and throwing objects at her. Neighbors intervened, and he was summoned again. When questioned, he twisted his actions as 'sharing joy and sorrow' with his wife, claiming hitting her brought him joy and her sorrow, and missing her brought the reverse. The judges were outraged and punished him appropriately.",
   'title': 'Sharing Joy and Sorrow',
   'url': 'https://fairytalez.com/sharing-joy-and-sorrow'}}]

In [34]:
objects = folktale["objects"]
places = folktale["places"]
agents = folktale["agents"]
relationships = folktale["relationships"]
events = folktale["events"]


In [35]:
def insert_entities(label: str, items: list[dict]):
    query = f"""
    MERGE (n:{label} {{id: $id}})
    SET n.type = $type,
		n.name = $name,
        n.description = $description
    RETURN n
    """

    for item in items:
        params = {
            "id": item["id"],
            "type": item["type"],
            "name": item["name"],
            "description": item["description"]
        }

        manager.execute_query(query, params)

insert_entities("Object", objects)

insert_entities("Place", places)


In [36]:
agent_query = """
MERGE (a:Agent {id: $agent_id})
SET a.race = $race,
	a.name = $name,
    a.ageGroup = $age_group,
    a.gender = $gender,
    a.description = $description

WITH a
MERGE (r:Role {name: $role})
MERGE (a)-[:HAS_ROLE]->(r)
"""

place_query = """
MATCH (a:Agent {id: $agent_id})
MATCH (p:Place {id: $place_id})
MERGE (a)-[:LIVES_IN]->(p)
"""

trait_query = """
MERGE (a:Agent {id: $agent_id})
MERGE (tr:Trait {name: $trait})
MERGE (a)-[r:HAS_TRAIT]->(tr)
SET r.strength = $strength
"""

for agent in agents:
    agent_id = agent["id"]

    agent["age_group"]

    params = {
        "agent_id": agent_id,
        "role": agent["role"],
        "race": agent["race"],
        "name": agent["name"],
        "age_group": agent["age_group"],
        "description": agent["description"],
        "gender": agent.get("gender")
    }

    manager.execute_query(agent_query, params)

    place_id = agent.get("lives_in")

    if place_id is not None:
        params = {
            "agent_id": agent_id,
            "place_id": place_id
        }

        manager.execute_query(place_query, params)

    personality = agent["personality"]
    for trait, strength in personality.items():
        params = {
            "agent_id": agent_id,
            "trait": trait,
            "strength": strength
        }

        manager.execute_query(trait_query, params)
    

In [37]:
print(relationships)

[{'source_id': 'agent-tailor', 'target_id': 'agent-wife', 'type': 'enemy', 'description': 'The tailor constantly abuses his wife', 'strength': 0.9}, {'source_id': 'agent-wife', 'target_id': 'agent-magistrate', 'type': 'enemy', 'description': 'The wife knows the magistrate and appeals to him for help', 'strength': 0.7}, {'source_id': 'agent-wife', 'target_id': 'agent-neighbours', 'type': 'knows', 'description': 'The wife relies on neighbors for support', 'strength': 0.7}]


In [38]:
relationship_query = """
MATCH (a:Agent {id: $source_id})
MATCH (b:Agent {id: $target_id})
MERGE (a)-[r:RELATIONSHIP {type: $type}]->(b)
SET r.description = $description,
    r.strength = $strength
"""

for rel in relationships:
    # print(rel)
    source_id = rel["source_id"]
    target_id = rel["target_id"]

    params = {
        "source_id": source_id,
        "target_id": target_id,
        "type": rel["type"],
        "description": rel["description"],
        "strength": rel["strength"]
    }
    print(params)

    manager.execute_query(relationship_query, params)


{'source_id': 'agent-tailor', 'target_id': 'agent-wife', 'type': 'enemy', 'description': 'The tailor constantly abuses his wife', 'strength': 0.9}
{'source_id': 'agent-wife', 'target_id': 'agent-magistrate', 'type': 'enemy', 'description': 'The wife knows the magistrate and appeals to him for help', 'strength': 0.7}
{'source_id': 'agent-wife', 'target_id': 'agent-neighbours', 'type': 'knows', 'description': 'The wife relies on neighbors for support', 'strength': 0.7}


In [39]:
event_query = """
MERGE (e:Event {id: $event_id})
SET e.description = $description,
	e.name = $name,
    e.order = $order

WITH e
MATCH (p:Place {id: $place_id})
MERGE (e)-[:TAKES_PLACE_IN]->(p)

WITH e
MERGE (fu:Function {name: $function})
MERGE (e)-[:HAS_FUNCTION]->(fu)
"""

agent_event_query = """
MATCH (e:Event {id: $event_id})
MATCH (a:Agent {id: $agent_id})
MERGE (e)-[r:HAS_AGENT]->(a)
SET r.importance = $importance,
    r.actions = $actions
"""

object_event_query = """
MATCH (e:Event {id: $event_id})
MATCH (o:Object {id: $object_id})
MERGE (e)-[:HAS_OBJECT]->(o)
"""

place_event_query = """
MATCH (e:Event {id: $event_id})
MATCH (p:Place {id: $place_id})
MERGE (e)-[:TAKES_PLACE_IN]->(p)
"""

for idx, event in enumerate(events):
    event_id = event["id"]
    place_id = event["place"]

    params = {
        "event_id": event_id,
        "description": event["description"],
        "name": event["name"],
        "order": idx,
        "place_id": place_id,
        "function": event["type"]
    }

    manager.execute_query(event_query, params)

    for agent in event["agents"]:
        params = {
            "event_id": event_id,
            "agent_id": agent["id"],
            "importance": agent["importance"],
            "actions": agent["action"]
        }

        manager.execute_query(agent_event_query, params)

    for object_id in event["objects"]:
        params = {
            "event_id": event_id,
            "object_id": object_id
        }

        manager.execute_query(object_event_query, params)


event_order_query = """
MATCH (e1:Event {id: $prev_id})
MATCH (e2:Event {id: $current_id})
MERGE (e1)-[:POST_EVENT]->(e2)
MERGE (e2)-[:PRE_EVENT]->(e1)
"""

for i in range(1, len(events)):
    params = {
        "prev_id": events[i-1]["id"],
        "current_id": events[i]["id"]
    }

    manager.execute_query(event_order_query, params)


first_event_query = """
MATCH (f:Folktale {url: $url})
MATCH (e:Event {id: $event_id})
MERGE (f)-[:FIRST_EVENT]->(e)
"""

params = {
    "url": folktale["url"],
    "event_id": events[0]["id"]
}

manager.execute_query(first_event_query, params)


[]

In [40]:
manager.close()
